# Trigger wave thresholds

In [1]:
import immunowave as iw
import numpy as np
import jax
import jax.numpy as jnp
import diffrax as dx

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from multiprocessing import Pool
import multiprocessing as mp
from functools import partial

import pickle

jax.config.update("jax_enable_x64", True)

from hill_function_utils import tissue_response, single_cell_response

import warnings
warnings.simplefilter("ignore")

In [2]:
%matplotlib qt

In [3]:
mpl.rc('axes', linewidth=4)
fontsize=24

def style_axes(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.xaxis.set_tick_params(labelsize=20)
    ax.yaxis.set_tick_params(labelsize=20)
    for tick in ax.xaxis.get_major_ticks():
        tick.label1.set_fontsize(fontsize)
    for tick in ax.yaxis.get_major_ticks():
        tick.label1.set_fontsize(fontsize)
    
    return ax



In [4]:
"""ok, so multiprocessing with jax only works if you use a 
non-default start method within the multiprocessing library. 
Here I use forkserver. An annoying corollary of forkserver 
(and also spawn) is that in jupyter notebooks, functions 
called within a Pool() need to be imported and can't be
defined in another cell. So I moved the functions to a
python file in this same directory."""
mp.set_start_method('forkserver')

## Model definition

Define the state and model for model 4 from the text.

In [5]:
class State4(iw.State):
    A: iw.ScalarField
    B: iw.ScalarField
        
        
class HillModel(iw.Model):
    KD: float
    n: int
    η: float
    ξ: float
    λ: float
    μ: float
    D: float=1.0
    gamma: float=1.0

    @jax.jit
    def __call__(self, t, state, args=None):
        # unpack field variables
        A, B = state.A, state.B
        # unpack parameters
        KD, n, η, ξ, λ, μ, D, gamma = self.KD, self.n, self.η, self.ξ, self.λ, self.μ, self.D, self.gamma
        # define PDE
        An = A.binop(n, jnp.power)
        tmp = An + KD ** n
        tmp = tmp.binop(-1, jnp.power)
        hill_term = An * tmp
        dAdt = D * A.laplacian(bc="neumann") + hill_term - gamma * A + η * B
        dBdt = ξ * B.laplacian(bc="neumann") + λ * B * (1 - B) - μ * A * B
        return State4(dAdt, dBdt)
    
    
#@jax.jit
def response(B0, KD, t_max, hill_coefficient=2):
    model = HillModel(KD=KD, n=hill_coefficient, η=1.0, ξ=0.0, λ=0.0, μ=0.0)
    state = State4(
        A=iw.ScalarField(shape, lb, h, 0),
        B=iw.ScalarField(
            shape, lb, h, fn=lambda x: B0 * jax.scipy.stats.norm.pdf(x, loc=L / 2, scale=1)
        ),
    )
    solution = iw.solve(
        model,
        state,
        t0=0,
        t1=t_max,
        t=jnp.array([t_max]),
        **kwargs,
    )
    return np.sum(solution.ys.A.values[-1] * solution.ys.A.h)
    # return solution.evaluate(t_final).A.integral() / L
    

def find_B0(final_mean_A, B0s):
    this_id = np.where(np.diff(final_mean_A) == np.max(np.diff(final_mean_A)))[0][0]
    B0c = 0.5 * (B0s[this_id] + B0s[this_id + 1])
    uncertainty = 0.5 * (B0s[this_id+1] - B0s[this_id])
    return B0c, uncertainty


"""these functions are now imported"""
# def tissue_response(B0, KD, t_max, hill_coefficient=2, D=1, gamma=1):
#     model = HillModel(KD=KD, n=hill_coefficient, η=1.0, ξ=0.0, λ=0.0, μ=0.0, D=D, gamma=gamma)
#     state = State4(
#         A=iw.ScalarField(shape, lb, h, 0),
#         B=iw.ScalarField(
#             shape, lb, h, fn=lambda x: B0 * jax.scipy.stats.norm.pdf(x, loc=L / 2, scale=1)
#         ),
#     )
#     solution = iw.solve(
#         model,
#         state,
#         t0=0,
#         t1=t_max,
#         t=jnp.array([t_max]),
#         **kwargs,
#     )
    
#     tissue_response = np.sum(solution.ys.A.values[-1] * solution.ys.A.h)
    
#     return tissue_response

    
# def single_cell_response(B0, KD, t_max, hill_coefficient=2, gamma=1.0):
#     """single cell response"""
#     model = HillModel(KD=KD, n=hill_coefficient, η=1.0, ξ=0.0, λ=0.0, μ=0.0, D=1.0, gamma=gamma)
#     state = State4(
#         A=iw.ScalarField((1,), lb, 1, 0),
#         B=iw.ScalarField(
#             (1,), lb, 1, fn=lambda x: B0 * jnp.array(x == 0).astype('float')
#         ),
#     )
#     solution = iw.solve(
#         model,
#         state,
#         t0=0,
#         t1=t_max,
#         t=jnp.array([t_max]),
#         **kwargs,
#     )
        
#     single_cell_response = solution.ys.A.values[-1, int(L // 2)]

#     return single_cell_response


'these functions are now imported'

## Plot critical bacteria concentrations as a function of various parameters for both tissue and single cell

### Vary KDs

In [6]:
KDs = np.logspace(-2, -0.5, 5)
B0s = np.logspace(-4, 0, 20)
t_max = 100
L = 100.0
n = 200
D = 1.0
gamma = 1.0
hill_coefficient = 4#[2, 3, 4, 5]
shape = (n,)
lb = [0]
h = L / (n - 1)
kwargs = dict(dt0=1e-4, max_steps=100000000, atol=1e-3, rtol=1e-3)


"""tissue"""
print('tissue')
final_integrated_A = np.zeros((len(KDs), len(B0s)), dtype=int)
for i, KD in enumerate(KDs):
    print(f"{KD=:.2f}")
    with Pool(processes=10) as pool:
        func = partial(tissue_response, KD=KD, t_max=t_max, 
                       hill_coefficient=hill_coefficient, D=D, gamma=gamma, shape=shape, 
                       lb=lb, h=h, L=L, **kwargs)    
        res = pool.map(func, B0s)
        final_integrated_A[i] = np.array(res)
      

"""single cell"""
print('single cell')
# run the model on a grid of length 1 with no diffusion
L = 1.0
n = 2
shape = (n,)
lb = [0]
h = L / (n - 1)

final_single_cell_A = np.zeros((len(KDs), len(B0s)), dtype=int)
for i, KD in enumerate(KDs):
    print(f"{KD=:.2f}")
    with Pool(processes=10) as pool:
        func = partial(single_cell_response, KD=KD, t_max=t_max, 
                       hill_coefficient=hill_coefficient, gamma=gamma, shape=shape, 
                       lb=lb, h=h, L=L, **kwargs)        
        res = pool.map(func, B0s)
        final_single_cell_A[i] = res
        
B0c_cell = np.zeros_like(KDs)
B0c_cell_uncertainties = np.zeros_like(KDs)
B0c_tissue = np.zeros_like(KDs)
B0c_tissue_uncertainties = np.zeros_like(KDs)
for i in range(len(KDs)):
    these_As = final_integrated_A[i]    
    B0c_tissue[i], B0c_tissue_uncertainties[i] = find_B0(these_As, B0s)
    
    these_As = final_single_cell_A[i]
    B0c_cell[i], B0c_cell_uncertainties[i] = find_B0(these_As, B0s)


tissue
KD=0.01


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

KD=0.02


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

KD=0.06


...

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

KD=0.13


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

KD=0.32


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

In [7]:
"""plot"""
plt.figure()
plt.errorbar(KDs, B0c_tissue, B0c_tissue_uncertainties, marker='o', markersize=18, markerfacecolor='none', 
             markeredgewidth=4, linestyle='none', markeredgecolor='g', elinewidth=4, capsize=6, ecolor='g', label='tissue numerics')
tissue_theory = 2 * (1 - (2 / (hill_coefficient + 1))) ** 0.5 * KDs ** (hill_coefficient / (hill_coefficient - 1))
plt.plot(KDs, tissue_theory, 'g-', linewidth=4, label='tissue theory')

plt.errorbar(KDs, B0c_cell, B0c_cell_uncertainties, marker='d', markersize=18, markerfacecolor='none', 
             markeredgewidth=4, linestyle='none', markeredgecolor='m', elinewidth=4, capsize=6, ecolor='m', label='single-cell numerics')

cell_theory = (1 - (1 / hill_coefficient)) * (1 / hill_coefficient) ** (1 / (hill_coefficient - 1)) * KDs ** (hill_coefficient / (hill_coefficient - 1))
plt.plot(KDs, cell_theory, 'm-', linewidth=4, label='single-cell theory')

plt.legend(fontsize=18)
plt.xscale("log")
plt.yscale("log")
plt.ylabel(r"$B_{0,c}$", fontsize=24)
plt.xlabel(r"concentration scale of positive feedback, $K_D$", fontsize=fontsize)
plt.minorticks_off()
ax = plt.gca()
style_axes(ax)
plt.tight_layout()
#plt.show()

"""plot ratio"""
plt.figure()
ratio = B0c_tissue / B0c_cell
ratio_uncertainty = ratio * np.sqrt((B0c_tissue / B0c_tissue_uncertainties) ** 2 + (B0c_cell / B0c_cell_uncertainties) ** 2 )
plt.errorbar(KDs, ratio, ratio_uncertainty, marker='o', markersize=18, markerfacecolor='none', 
             markeredgewidth=4, linestyle='none', markeredgecolor='c', elinewidth=4, capsize=6, ecolor='c', label='numerics')

# theoretical ratio
KD_theory = np.logspace(np.log10(np.min(KDs)), np.log10(np.max(KDs)), 1000)
ratio_theory = 2 * (1 -(2 / (hill_coefficient +1))) ** (1/2) / ((1 - 1/hill_coefficient)*(1/hill_coefficient)**(1/(hill_coefficient-1))) * np.sqrt(D / gamma) * np.ones_like(KD_theory)
plt.plot(KD_theory, ratio_theory, 'k-', linewidth=4, label='theory')
plt.xlabel(r"concentration scale of positive feedback, $K_D$", fontsize=fontsize)
plt.ylabel('tissue/cell threshold ratio', fontsize=fontsize)
plt.legend(fontsize=18)
style_axes(plt.gca())
plt.tight_layout()

### Vary hill coefficients

In [10]:
"""vary hill coefficients"""
KD = 0.01#np.logspace(-2, -0.5, 5)
B0s = np.logspace(-6, 0, 20)
t_max = 100
L = 100.0
n = 200
hill_coefficients = [2, 3, 4, 5]
shape = (n,)
lb = [0]
h = L / (n - 1)
kwargs = dict(dt0=1e-4, max_steps=100000000, atol=1e-3, rtol=1e-3)


"""tissue"""
print('tissue')
final_integrated_A = np.zeros((len(hill_coefficients), len(B0s)), dtype=int)
for i, hill_coefficient in enumerate(hill_coefficients):
    print(f"{hill_coefficient=:.2f}")
    with Pool(processes=10) as pool:
        func = partial(tissue_response, KD=KD, t_max=t_max, 
                       hill_coefficient=hill_coefficient, D=D, gamma=gamma, shape=shape, 
                       lb=lb, h=h, L=L, **kwargs)
        res = pool.map(func, B0s)
        final_integrated_A[i] = np.array(res)
      

"""single cell"""
print('single cell')
# run the model on a grid of length 1 with no diffusion
L = 1.0
n = 2
shape = (n,)
lb = [0]
h = L / (n - 1)

final_single_cell_A = np.zeros((len(hill_coefficients), len(B0s)), dtype=int)
for i, hill_coefficient in enumerate(hill_coefficients):
    print(f"{hill_coefficient=:.2f}")
    with Pool(processes=10) as pool:
        func = partial(single_cell_response, KD=KD, t_max=t_max, 
                       hill_coefficient=hill_coefficient, gamma=gamma, shape=shape, 
                       lb=lb, h=h, L=L, **kwargs)
        res = pool.map(func, B0s)
        final_single_cell_A[i] = res
        
"""compute critical bacteria thresholds"""        
B0c_cell = np.zeros(len(hill_coefficients))
B0c_cell_uncertainties = np.zeros(len(hill_coefficients))
B0c_tissue = np.zeros(len(hill_coefficients))
B0c_tissue_uncertainties = np.zeros(len(hill_coefficients))

for i in range(len(hill_coefficients)):
    these_As = final_integrated_A[i]    
    B0c_tissue[i], B0c_tissue_uncertainties[i] = find_B0(these_As, B0s)
    
    these_As = final_single_cell_A[i]
    B0c_cell[i], B0c_cell_uncertainties[i] = find_B0(these_As, B0s)

tissue
hill_coefficient=2.00


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

hill_coefficient=3.00


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

hill_coefficient=4.00


...

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

hill_coefficient=4.00


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

hill_coefficient=5.00


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

In [12]:
"""plot"""
hill_coefficients = np.array(hill_coefficients)
ns = np.linspace(2, 5, 100)

plt.figure()
plt.errorbar(hill_coefficients, B0c_tissue, B0c_tissue_uncertainties, marker='o', markersize=18, markerfacecolor='none', 
             markeredgewidth=4, linestyle='none', markeredgecolor='g', elinewidth=4, capsize=6, ecolor='g', label='tissue numerics')
tissue_theory = 2 * (1 - (2 / (ns + 1))) ** 0.5 * KD ** (ns / (ns - 1))
plt.plot(ns, tissue_theory, 'g-', linewidth=4, label='tissue theory')

plt.errorbar(hill_coefficients, B0c_cell, B0c_cell_uncertainties, marker='d', markersize=18, markerfacecolor='none', 
             markeredgewidth=4, linestyle='none', markeredgecolor='m', elinewidth=4, capsize=6, ecolor='m', label='single-cell numerics')

cell_theory = (1 - (1 / ns)) * (1 / ns) ** (1 / (ns - 1)) * KD ** (ns / (ns - 1))
plt.plot(ns, cell_theory, 'm-', linewidth=4, label='single-cell theory')

plt.legend(fontsize=18)
#plt.xscale("log")
plt.yscale("log")
plt.ylabel(r"$B_{0,c}$", fontsize=24)
plt.xlabel(r"hill coefficient, $n$", fontsize=24)
#plt.xlim([2e-3, 10 ** (-0.5)])
#plt.ylim([1e-5, 1e1])
plt.minorticks_off()
ax = plt.gca()
style_axes(ax)
plt.tight_layout()
#plt.show()

# plot tissue-cell ratio


plt.figure()

ratio = B0c_tissue / B0c_cell
ratio_uncertainty = ratio * np.sqrt((B0c_tissue / B0c_tissue_uncertainties) ** 2 + (B0c_cell / B0c_cell_uncertainties) ** 2 )
plt.errorbar(hill_coefficients, ratio, ratio_uncertainty, marker='o', markersize=18, markerfacecolor='none', 
             markeredgewidth=4, linestyle='none', markeredgecolor='c', elinewidth=4, capsize=6, ecolor='c', label='numerics')

# theoretical ratio
ns = np.linspace(2, 5, 100)
ratio_theory = 2 * (1 -(2 / (ns +1))) ** (1/2) / ((1 - 1/ns)*(1/ns)**(1/(ns-1))) * np.sqrt(D / gamma)
plt.plot(ns, ratio_theory, 'k-', linewidth=4, label='theory')
plt.xlabel(r"hill coefficient, $n$", fontsize=24)
plt.ylabel('tissue/cell threshold ratio')
plt.legend(fontsize=18)
style_axes(plt.gca())
plt.tight_layout()



### Vary decay rate

In [13]:
"""vary decay rate. now time in minutes. length in microns"""
KD = 0.01#np.logspace(-2, -0.5, 5)
B0s = np.logspace(-6, 0, 20)
t_max = 100
L = 100.0
n = int(L)
hill_coefficient = 3
gammas = np.linspace(0.02, 0.2, 5)
D = 60.0      
shape = (n,)
lb = [0]
h = L / (n - 1)
kwargs = dict(dt0=1e-4, max_steps=100000000, atol=1e-3, rtol=1e-3)


"""tissue"""
print('tissue')
final_integrated_A = np.zeros((len(gammas), len(B0s)), dtype=int)
for i, gamma in enumerate(gammas):
    print(f"{gamma=:.2f}")
    with Pool(processes=10) as pool:
        func = partial(tissue_response, KD=KD, t_max=t_max, 
                       hill_coefficient=hill_coefficient, D=D, gamma=gamma, shape=shape, 
                       lb=lb, h=h, L=L, **kwargs)
        res = pool.map(func, B0s)
        final_integrated_A[i] = np.array(res)
      

"""single cell"""
print('single cell')
# run the model on a grid of length 1 with no diffusion
L = 1.0
n = 2
shape = (n,)
lb = [0]
h = L / (n - 1)

final_single_cell_A = np.zeros((len(gammas), len(B0s)), dtype=int)
for i, gamma in enumerate(gammas):
    print(f"{gamma=:.2f}")
    with Pool(processes=10) as pool:
        func = partial(single_cell_response, KD=KD, t_max=t_max, hill_coefficient=hill_coefficient, 
                       gamma=gamma, shape=shape, lb=lb, h=h, L=L, **kwargs)
        res = pool.map(func, B0s)
        final_single_cell_A[i] = res
        
"""compute critical bacteria thresholds"""        
B0c_cell = np.zeros(len(gammas))
B0c_cell_uncertainties = np.zeros(len(gammas))
B0c_tissue = np.zeros(len(gammas))
B0c_tissue_uncertainties = np.zeros(len(gammas))

for i in range(len(gammas)):
    these_As = final_integrated_A[i]    
    B0c_tissue[i], B0c_tissue_uncertainties[i] = find_B0(these_As, B0s)
    
    these_As = final_single_cell_A[i]
    B0c_cell[i], B0c_cell_uncertainties[i] = find_B0(these_As, B0s)

tissue
gamma=0.02


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

gamma=0.07


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

gamma=0.11


...

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

gamma=0.15


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

gamma=0.20


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

In [15]:
"""plot"""

gamma_theory = np.linspace(np.min(gammas), np.max(gammas), 1000)

plt.figure()
plt.errorbar(gammas, B0c_tissue, B0c_tissue_uncertainties, marker='o', markersize=18, markerfacecolor='none', 
             markeredgewidth=4, linestyle='none', markeredgecolor='g', elinewidth=4, capsize=6, ecolor='g', label='tissue numerics')
tissue_theory = 2 * (1 - (2 / (hill_coefficient + 1))) ** 0.5 * (KD * gamma_theory) ** (hill_coefficient / (hill_coefficient - 1)) * np.sqrt(D / gamma_theory)
plt.plot(gamma_theory, tissue_theory, 'g-', linewidth=4, label='tissue theory')

plt.errorbar(gammas, B0c_cell, B0c_cell_uncertainties, marker='d', markersize=18, markerfacecolor='none', 
             markeredgewidth=4, linestyle='none', markeredgecolor='m', elinewidth=4, capsize=6, ecolor='m', label='single-cell numerics')

cell_theory = (1 - (1 / hill_coefficient)) * (1 / hill_coefficient) ** (1 / (hill_coefficient - 1)) * (KD * gamma_theory) ** (hill_coefficient / (hill_coefficient - 1))
plt.plot(gamma_theory, cell_theory, 'm-', linewidth=4, label='single-cell theory')

plt.legend(fontsize=18)
#plt.xscale("log")
plt.yscale("log")
plt.ylabel(r"$B_{0,c}$", fontsize=24)
plt.xlabel(r"decay rate, $\gamma$ (1/min)", fontsize=24)
#plt.xlim([2e-3, 10 ** (-0.5)])
#plt.ylim([1e-5, 1e1])
plt.minorticks_off()
ax = plt.gca()
style_axes(ax)
plt.tight_layout()
#plt.show()


# plot tissue-cell ratio

plt.figure()

ratio = B0c_tissue / B0c_cell
ratio_uncertainty = ratio * np.sqrt((B0c_tissue / B0c_tissue_uncertainties) ** 2 + (B0c_cell / B0c_cell_uncertainties) ** 2 )
plt.errorbar(gammas, ratio, ratio_uncertainty, marker='o', markersize=18, markerfacecolor='none', 
             markeredgewidth=4, linestyle='none', markeredgecolor='c', elinewidth=4, capsize=6, ecolor='c', label='numerics')

# theoretical ratio
ratio_theory = 2 * (1 -(2 / (hill_coefficient +1))) ** (1/2) / ((1 - 1/hill_coefficient)*(1/hill_coefficient)**(1/(hill_coefficient-1))) * np.sqrt(D / gamma_theory)
plt.plot(gamma_theory, ratio_theory, 'k-', linewidth=4, label='theory')
plt.xlabel(r"decay rate, $\gamma$ (1/min)", fontsize=fontsize)
plt.ylabel('tissue/cell threshold ratio', fontsize=fontsize)
plt.legend(fontsize=18)
style_axes(plt.gca())
plt.tight_layout()